<a href="https://colab.research.google.com/github/filippopedrini95/MS_DF_ComputerVision_Project/blob/main/ComputerVisionProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Libraries

In [1]:
import pandas as pd
import numpy as np

import keras
import tensorflow as tf

from sklearn.model_selection import train_test_split

from keras.applications.resnet50 import preprocess_input as resnet_preprocess

from keras.models import Sequential
from keras.layers import Dense, Dropout

from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.datasets import cifar10

from sklearn.metrics import classification_report

from pathlib import Path

I0000 00:00:1773275942.910790   76188 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773275942.914049   76188 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773275943.664497   76188 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773275947.612367   76188 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

#Import Dataset & Images Preprocessing

In [2]:
#import image dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

/workspaces/MS_DF_ComputerVision_Project/.venv/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


## ResNet50

### image preprocessing for ResNet

In [3]:
#image preprocessing to match ResNet input requirements
x_test_resnet = resnet_preprocess(x_test)

In [4]:
#image preprocessing to match ResNet input requirements
x_train_resnet = resnet_preprocess(x_train)

In [5]:
#validation set
x_train_resnet, x_val, y_train_resnet, y_val = train_test_split(
    x_train_resnet,
    y_train,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y_train
)

In [6]:
y_val

array([[2],
       [7],
       [1],
       ...,
       [5],
       [9],
       [0]], shape=(10000, 1), dtype=uint8)

### No Feature Extractor Optimization

In [7]:
#import ResNet50 for feature extraction
pretrained_model = keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(32,32,3),
    pooling="max",
    name="resnet50",
)

pretrained_model.trainable = False # disable ResNet50 parameter optimization

#define the structure of the CNN
cnn_model = Sequential([
    pretrained_model,
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])

#compile the model
cnn_model.compile(optimizer="adam",
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

#define early stopping
early_stop = EarlyStopping(
    monitor="val_loss",         # monitors validation loss function
    patience=4,                 # number of steps with no improvements in "val_loss" before stopping the fitting
    restore_best_weights=True   # in case the further "patience" steps made the performance worse, it restores the best weights
)

E0000 00:00:1773275956.665584   76188 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
#model fitting
history = cnn_model.fit(
    x=x_train_resnet,
    y=y_train_resnet,
    epochs= 1, #20,
    batch_size=64,
    validation_data=(x_val, y_val),
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

625/625 ━━━━━━━━━━━━━━━━━━━━ 160s 248ms/step - accuracy: 0.5440 - loss: 1.3445 - val_accuracy: 0.6155 - val_loss: 1.0970


In [9]:
#save the model after training
#cnn_model.save("imgclf_resnet50_cifar10_v1.keras")
print("model saved")

model saved


In [10]:
#model prediction
y_pred_probs = cnn_model.predict(x_test_resnet)
y_pred = np.argmax(y_pred_probs, axis=1)

#model evaluation parameters
test_loss, test_accuracy = cnn_model.evaluate(x_test_resnet, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 111ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 106ms/step - accuracy: 0.6161 - loss: 1.0985


In [11]:
print("MODEL PERFORMANCE ON TEST SET")
print("accuracy:",test_accuracy)
print("loss:",test_loss, end="\n\n")

#classification report
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print(classification_report(y_test, y_pred, target_names=class_names))

MODEL PERFORMANCE ON TEST SET
accuracy: 0.616100013256073
loss: 1.0984529256820679

              precision    recall  f1-score   support

    airplane       0.64      0.66      0.65      1000
  automobile       0.67      0.65      0.66      1000
        bird       0.64      0.46      0.54      1000
         cat       0.46      0.54      0.49      1000
        deer       0.59      0.50      0.54      1000
         dog       0.61      0.55      0.58      1000
        frog       0.67      0.72      0.69      1000
       horse       0.56      0.72      0.63      1000
        ship       0.73      0.65      0.69      1000
       truck       0.64      0.71      0.67      1000

    accuracy                           0.62     10000
   macro avg       0.62      0.62      0.62     10000
weighted avg       0.62      0.62      0.62     10000

